In [1]:
import warnings

warnings.filterwarnings('ignore')

# Data Processing
from sklearn.model_selection import train_test_split

# Deep Learning
from tensorflow.keras.preprocessing.sequence import pad_sequences

# NLP
import nltk

# Local Modules
from models.xgboost_model import XGBoostModel
from models.text_cnn_model import TextCNNModel
from modules.data_loader import DataLoader
from modules.fake_news_detector import FakeNewsDetector
from modules.feature_extractor import FeatureExtractor
from modules.model_evaluator import ModelEvaluator
from modules.text_processor import TextPreprocessor

In [2]:
# Download required NLTK data
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4', quiet=True)

# Required for POS tagging
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print("=" * 80)
print("FAKE NEWS DETECTION SYSTEM - WELFake Dataset")
print("=" * 80)

FAKE NEWS DETECTION SYSTEM - WELFake Dataset


In [3]:
DATA_PATH = 'files/WELFake_Dataset.csv'
GLOVE_PATH = 'files/glove.6B.100d.txt'

# Model settings
TRAIN_XGBOOST = True
TRAIN_TEXT_CNN = True  # I have glove embeddings, so I am setting this to `True`
TUNE_HYPERPARAMETERS = False  # If I set this to True, it will perform grid search (slower)
USE_POS_FEATURES = True  # Enable POS tagging features for improved accuracy

In [4]:
# Load and explore data
loader = DataLoader()
df = loader.load_data(DATA_PATH)

df.head()


LOADING DATA...
--------------------------------------------------------------------------------
  Dataset loaded successfully
  Shape: (72134, 4)
  Columns: ['Unnamed: 0', 'title', 'text', 'label']

  Label distribution:
    Fake news (0): 35028
    Legit news (1): 37106

  Missing values detected:
title    558
text      39
dtype: int64


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [ ]:
preprocessor = TextPreprocessor(use_pos_features=USE_POS_FEATURES)
df = preprocessor.preprocess_dataframe(df)

print("Original text:", df['text'].iloc[0][:200])
print("\nCleaned text:", df['combined'].iloc[0][:200])


PREPROCESSING DATA...
--------------------------------------------------------------------------------
  Cleaning title...
  Cleaning text...
  Combining title and text...
  Extracting POS features (this may take a while)...


In [ ]:
X = df['combined'].values
y = df['label'].values

# Initialize POS feature DataFrames
df_train_pos = None
df_val_pos = None
df_test_pos = None

# Keep DataFrame indices for POS features
df_indices = df.index.values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_indices, test_size=0.2, random_state=5, stratify=y
)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_train, y_train, idx_train, test_size=0.125, random_state=5, stratify=y_train
)

# Get POS features DataFrames for each split (if using POS features)
if USE_POS_FEATURES:
    df_train_pos = df.loc[idx_train]
    df_val_pos = df.loc[idx_val]
    df_test_pos = df.loc[idx_test]

print(f"Training: {len(X_train)} samples, Validation: {len(X_val)} samples, Testing: {len(X_test)} samples")

In [ ]:
results = {}
xgb_model = None
feature_extractor_xgb = None

In [ ]:
# Train XGBoost model
feature_extractor_xgb = FeatureExtractor()

# Extract TF-IDF features
X_train_tfidf = feature_extractor_xgb.create_tfidf_features(X_train)
X_val_tfidf = feature_extractor_xgb.tfidf_vectorizer.transform(X_val)
X_test_tfidf = feature_extractor_xgb.tfidf_vectorizer.transform(X_test)

# Apply SVD
X_train_svd = feature_extractor_xgb.apply_svd(X_train_tfidf)
X_val_svd = feature_extractor_xgb.svd.transform(X_val_tfidf)
X_test_svd = feature_extractor_xgb.svd.transform(X_test_tfidf)

# Combine with POS features if enabled
if USE_POS_FEATURES:
    X_train_final = feature_extractor_xgb.combine_features(X_train_svd, df_train_pos)
    X_val_final = feature_extractor_xgb.combine_features(X_val_svd, df_val_pos)
    X_test_final = feature_extractor_xgb.combine_features(X_test_svd, df_test_pos)
else:
    X_train_final = X_train_svd
    X_val_final = X_val_svd
    X_test_final = X_test_svd

# Train XGBoost
xgb_model = XGBoostModel()
xgb_model.train(
    X_train_final, y_train,
    X_val_final, y_val,
    tune_hyperparameters=TUNE_HYPERPARAMETERS
)

In [ ]:
# Evaluate XGBoost
y_pred_xgb = xgb_model.predict(X_test_final)
y_pred_proba_xgb = xgb_model.predict_probabilities(X_test_final)

results['XGBoost'] = ModelEvaluator.evaluate(
    y_test, y_pred_xgb, y_pred_proba_xgb, "XGBoost"
)

results_xgb = ModelEvaluator.evaluate(y_test, y_pred_xgb, y_pred_proba_xgb, "XGBoost")
ModelEvaluator.plot_confusion_matrix(y_test, y_pred_xgb, "XGBoost")
ModelEvaluator.plot_roc_curve(y_test, y_pred_proba_xgb, "XGBoost")

In [ ]:
feature_extractor_cnn = FeatureExtractor()

# Prepare sequences
X_train_seq, vocab_size = feature_extractor_cnn.prepare_sequences(X_train)
X_val_seq = pad_sequences(
    feature_extractor_cnn.tokenizer.texts_to_sequences(X_val),
    maxlen=500, padding='post', truncating='post'
)
X_test_seq = pad_sequences(
    feature_extractor_cnn.tokenizer.texts_to_sequences(X_test),
    maxlen=500, padding='post', truncating='post'
)

# Try to load GloVe embeddings
try:
    embedding_matrix = feature_extractor_cnn.load_glove_embeddings(
        GLOVE_PATH, embedding_dim=100
    )
except (FileNotFoundError, IOError, ValueError):
    embedding_matrix = None
    print("  Continuing without pre-trained embeddings")

# Build and train TextCNN
text_cnn = TextCNNModel(
    vocab_size=vocab_size,
    embedding_dim=100,
    max_len=500,
    embedding_matrix=embedding_matrix
)

text_cnn.build_model()
text_cnn.train(X_train_seq, y_train, X_val_seq, y_val, epochs=10)

In [ ]:
 # Evaluate TextCNN
y_pred_cnn = text_cnn.predict(X_test_seq)
y_pred_proba_cnn = text_cnn.predict_probabilities(X_test_seq)

results['TextCNN'] = ModelEvaluator.evaluate(
    y_test, y_pred_cnn, y_pred_proba_cnn, "TextCNN"
)

results_cnn = ModelEvaluator.evaluate(y_test, y_pred_cnn, y_pred_proba_cnn, "TextCNN")
ModelEvaluator.plot_confusion_matrix(y_test, y_pred_cnn, "TextCNN")
ModelEvaluator.plot_roc_curve(y_test, y_pred_proba_cnn, "TextCNN")

In [ ]:
# Compare XGBoost vs TextCNN
results = {'XGBoost': results_xgb, 'TextCNN': results_cnn}
comparison = ModelEvaluator.compare_models(results)

In [ ]:
# Test on custom inputs
detector_xgb = FakeNewsDetector(xgb_model, preprocessor, feature_extractor_xgb, 'xgboost', use_pos_features=USE_POS_FEATURES)

# Test FAKE example
fake_title = "EXPOSED: Hillary Clinton Caught Hiding Alien Discovery On Pluto In Secret Cover-Up"
fake_text = """
Hillary Clinton and the corrupt Democrats have been caught in another massive cover-up. Leaked emails prove that crooked Hillary personally ordered NASA to hide proof of alien life discovered on Pluto from the American people.

The lying mainstream media refuses to report on this bombshell story. CNN and the failing New York Times are completely silent because they are controlled by George Soros and the globalist elites who don't want you to know the truth.

Whistleblowers inside NASA confirm that Barack Obama was briefed on the alien discovery but chose to lie to the American public. This is the same Obama administration that weaponized the FBI and DOJ against President Trump.

Wake up America! The deep state has been hiding extraterrestrial contact for decades. Share this before Facebook censors it and Big Tech tries to silence the truth. The radical left will stop at nothing to keep you in the dark.

This is bigger than Benghazi. Hillary must be locked up for her crimes against humanity. The corrupt establishment is terrified that patriots are finally exposing their lies.
"""

print("=" * 80)
print("FAKE NEWS TEST - XGBoost Model")
print("=" * 80)
result_xgb_fake = detector_xgb.predict_article(fake_title, fake_text, verbose=True)
print(f"Prediction: {result_xgb_fake['prediction']}")
print(f"Confidence: {result_xgb_fake['confidence']:.2%}")

In [ ]:
# Test FAKE example with TextCNN
detector_cnn = FakeNewsDetector(text_cnn, preprocessor, feature_extractor_cnn, 'textcnn')

print("=" * 80)
print("FAKE NEWS TEST - TextCNN Model")
print("=" * 80)
result_cnn_fake = detector_cnn.predict_article(fake_title, fake_text, verbose=True)
print(f"Prediction: {result_cnn_fake['prediction']}")
print(f"Confidence: {result_cnn_fake['confidence']:.2%}")

In [ ]:
# Test LEGIT example from CNN, accessed 20th December 2025 @ https://edition.cnn.com/2025/12/19/health/parents-of-adults-facing-addiction
legit_title = "‘A lifelong struggle’: What it’s like for some parents of adults facing addiction"
legit_text = """
The deaths of Hollywood director Rob Reiner and his producer wife, Michele, set off waves of tributes and memorials, but for some families, it stopped them cold.

Nick Reiner, the couple’s 32-year-old son, has been charged in their deaths. He has been open about his drug addiction and several stints in rehab, and his parents had spoken about trying to do everything they could to help him.

Although few details about the Reiners’ deaths have been released, their history feels especially familiar to parents of people with addiction problems.


“We’ve been pretty much glued to the news about it, and it’s been kind of tough. Far too often, bad things happen in families when there’s someone addicted to drugs,” said Ron Grover of Lake of the Ozarks, Missouri.

His son’s drug abuse took over not just his life, but the entire family’s life, for years.

“This is not a disease that affects just one person. It’s a family disease,” said Grover, who said his son is now sober.

And while many people think this couldn’t happen to their child or their loved one, drug addiction cuts across race and class, and it is a problem in a wide variety of communities, even if stigma often keeps people from openly talking about it. More than 48.4 million Americans have a substance use disorder, according to government statistics, and many millions more family members are trying to find the best way to help their loved one.


Parricide — the killing of a parent — is extremely rare, but the chaos, unpredictability and constant worry are common sentiments among the parents who spoke with CNN about what it’s like to have children with drug addictions.

Related article
Nick Reiner attends the Los Angeles Premiere of ''Spinal Tap II: The End Continues'' at The Egyptian Theatre in Los Angeles, California, on September 9, 2025.
What we know about Nick Reiner, son of Rob and Michele Reiner

After Marilynn Ulrich of Connecticut learned about the Reiners’ deaths, she posted in a Facebook group for parents of children with addiction issues and said she too had feared for her safety “because I don’t know what my addicted son is capable of.” Her post generated a flurry of comments from parents who said they too were afraid for — and sometimes afraid of — their children.

“It’s a real concern. I was talking about it with my husband last night. That’s why this story is so upsetting, particularly since, like so many parents, they tried everything they could do to help,” Ulrich told CNN of the Reiners.

Ulrich doesn’t see her son, whose name she asked not to share, on a regular basis because he lives in California. Over the years, she has tried to get help for him.


“As a mother, you’re looking for answers, because you want to solve this problem. It’s hard to just say there’s no answers and I can’t solve it as a mother,” Ulrich said. “It never leaves your mind. It’s always hovering, and you’re wondering if you’re going to get a phone call about some calamity.”

Having also worked in rehab, she says, she knows treatment can help, but “you can’t force your kid to get well.” And sometimes, multiple trips to rehab are needed.

Unlike other diseases where a treatment can reliably keep someone stable, it’s difficult to know what will work with drug addiction. There are a variety of approaches and no simple one-size-fits-all solutions, said Dr. Kimberly Kirby, a professor of psychology at Rowan University in New Jersey. “And the things the family goes through, it’s just heartbreak. It’s not just the heartbreak of seeing a child in trouble but laying out thousands of dollars, depleting retirement funds trying to save their child.”

There certainly have been successes with treatments and family interventions, she said, but not everyone reaches a happy ending.

‘I don’t give up’
Audrey Bonafe, who lives just south of Richmond, Virginia, says her son Foster is clean now, but it was a long and harrowing journey to sobriety. She’ll never forget the night when he turned up at her house with a gun, she says, and was high on something.


“My son had an AR-15 in his hand and said that there was a pregnant woman that was in the driveway, and he was going to protect her,” Bonafe said. But Foster was hallucinating. There was no one in the driveway.

Fortunately, he didn’t hurt anyone with the gun, but he did cut himself.

“There was blood all over. I just kept trying to stop the blood, but he just wouldn’t be still,” she said. “It was just awful.”

Related article
20250206-fentanyl-explainer-card___exlargetease.png
Fentanyl in the US: A visual guide

Like the Reiners, Bonafe and her husband got their son into rehab multiple times – she says they spent their retirement savings to do it – but it never fully worked. He was not violent with them, she said, but he stole, and she constantly worried.


Life with an addiction was dangerous and difficult for Foster. At one point, he ended up in jail. He lived in his car, and he was shot in a drug deal that cost him one finger and the use of others.

Eventually, what got Foster clean was a long stint in the hospital for a blood infection. It was during the Covid-19 pandemic, and the visitor restrictions meant he couldn’t access illicit drugs. When he finally got out after eight months, a doctor who had grown to like him handed him a note that said, “don’t f*ck it up.” Bonafe gave him a bracelet with the same message for Christmas, and he wears it every day.

Things have been going well so far, but life isn’t perfect. Her doggy day care business is going strong, but she worries about money. Her husband has had a brain tumor, and her son’s blood infection left him with a weak heart.

“It has been a struggle, but I don’t give up easily,” she said, but Instead, she’s giving back: In addition to Gigi’s Place, a nonprofit that cares for pets when people are hospitalized, she started Mother of an Addict, a Facebook group with thousands of members. Her son has also been posting about his experience, and they hope to start a podcast to help even more families dealing with addiction.

‘A lifelong struggle’
Ron and Darlene Grover’s son got sober in July 2010. He now has a good job, a family and a home, they said, and they trust and believe in him, but they know that sobriety is a constant struggle for most people.


“I hope it never comes back, but the reality is, it’s a lifelong struggle for the addict and a lifelong struggle for the loved ones, too,” Ron Grover said.

Their son, whom they asked not to name, started using drugs when he was 15. They found out when he got caught shoplifting. It seemed so unlike him – he was a good kid and a type A personality, his dad said – so they dug into why he did it and learned that he had a drug problem and needed to feed his habit.

Related video
20250324-fentanyl_split emt fentanyl needle_exlarge16x9.jpg
video
CNN’s two-year investigation into the US’s fentanyl crisis: ‘I never really understood what a drug addict was’

During the seven years he was using, the Grovers said, they tried everything: Narcotics Anonymous meetings, a therapist, four rehab facilities. Nothing worked, and his addiction led him to all five jails in Kansas City, as well as the state prison, Ron Grover said.


A hospital called in the middle of the night to say that their son was unconscious and not breathing – “these kind of phone calls never leave you,” Grover said.

His son finally quit cold turkey.

Today, Grover speaks to a wide variety of audiences about what it’s like to be a parent of a child with an addiction. He’s kept an extensive blog about it, and it’s made an impact: One essay about the seven truths of his son’s addiction that took five years to learn, has been shared by professors and counselors.

“We spent years trying to fix things, but you can’t fix an addict. The best thing to do is fix yourself so that you can help them,” Grover said. “Finally, I came to the realization that if I want to do something, I can do my best to try to keep him alive until the day recovery comes, and that’s what I focused on. I think it helped me and helped him, too.”

Parents can have blinders on
In 2018, Kim Humphrey became the first full-time staff member of Parents of Addicted Loved Ones, a free resource to help parents of people with substance use issues. In all those years of facilitating meetings, he’s seen parents try everything to help but still lose their children to overdose.


“I’ve seen multiple cases, too, where violence has been involved,” said Humphrey, who had a 32-year career with the Phoenix police department. One couple now in his group had to get a protection order against their son. “It’s so sad, because they so want to so badly help them, but they started to experience violence.”

The group is helpful in that situation, he said, because parents can sometimes have “their blinders up” about their child, and they don’t protect themselves.



still_22202969_23147.262000000002_thumb.jpg
Reiner family friend recalls staying with them weeks ago
4:26
There’s a lot of misinformation about what it’s like to have a child with an addiction problem, he says. When he told some friends about his two sons’ addiction problems, the first thing they asked was, “What do you think you did wrong?”


Too many people see addiction as a moral failure or believe drug users weren’t raised in a loving home, Humphrey says. Yet his sons were raised in a loving home, and both ended up on the streets, in hospitals and in jail while they were using.

None of that was because they didn’t know right from wrong, he said.

“This is about a brain disorder. These drugs completely change and rewire them to the point where their decision-making part is affected,” Humphrey said.

There’s stigma, too, that makes it difficult for people to find or ask for help, said Dr. Patrick Corrigan, director of the Center for Health Equity, Education and Research, who writes extensively about stigma and empowerment around drug addiction and mental illness. “The problem with stigmatized situations is, people aren’t going to talk about it, and because the stigma is so bad, it’s hard for family and friends to figure out what to do.”


Humphrey learned that although he couldn’t fix things for his children, there were healthy things he could do: Treat them like adults. Realize there are consequences to their actions. Be a cheerleader. Support recovery. Do everything possible to get them the help they need, and realize that you can’t make them do anything.

“It’s not hard to understand. It’s hard to do,” he said.

Related article
Tatro fixed 2.jpg
A mother turns grief over her daughter’s overdose death into a movement to save others struggling with addiction

Both of Humphrey’s sons finally got sober. For his oldest, it happened when he ended up in the hospital and his dad told the workers to say that he loved his son very much but wasn’t coming to get him again.


“My son said he laid there in that hospital bed, and it occurred to him that he didn’t want to lose those relationships, and he didn’t want this, and he had been through it so many times,” Humphrey said. He got into a treatment program and then went to find his brother, who also eventually got into a program.

Both sons are now married. They own homes and have good jobs. And now, Humphrey has grandchildren.

“Is everything perfect in our life? No, it’s life, but the reality is that we were sure we’d never see them alive or that they’d be in prison for the rest of their lives, and that’s not the case,” Humphrey said.

His group, Parents of Addicted Loved Ones, has expanded from a handful of gatherings in Arizona to providing help with meetings in 40 states, online and in 11 foreign countries.

“I get these messages of hope every day. People do get better. What we do is help people focus on the fact that their child is alive, and if they’re breathing, there’s hope,” he said. “Let’s do what we can to try and help them, but in the meantime, let’s also take care of ourselves, because we’re kind of worthless if we can’t function.”
"""

print("=" * 80)
print("LEGIT NEWS TEST - XGBoost Model")
print("=" * 80)
detector_xgb = FakeNewsDetector(xgb_model, preprocessor, feature_extractor_xgb, 'xgboost', use_pos_features=USE_POS_FEATURES)
result_xgb_legit = detector_xgb.predict_article(legit_title, legit_text, verbose=True)
print(f"Prediction: {result_xgb_legit['prediction']}")
print(f"Confidence: {result_xgb_legit['confidence']:.2%}")

In [ ]:
# Test LEGIT example with TextCNN
print("=" * 80)
print("LEGIT NEWS TEST - TextCNN Model")
print("=" * 80)
result_cnn_legit = detector_cnn.predict_article(legit_title, legit_text, verbose=True)
print(f"Prediction: {result_cnn_legit['prediction']}")
print(f"Confidence: {result_cnn_legit['confidence']:.2%}")

In [ ]:
# Compare predictions on both examples
print("=" * 80)
print("COMPARISON SUMMARY")
print("=" * 80)
print("\nFAKE NEWS TEST:")
print(f"  Title: {fake_title[:60]}...")
print(f"  XGBoost: {result_xgb_fake['prediction']} ({result_xgb_fake['confidence']:.2%} confidence)")
print(f"  TextCNN: {result_cnn_fake['prediction']} ({result_cnn_fake['confidence']:.2%} confidence)")

print("\nLEGIT NEWS TEST:")
print(f"  Title: {legit_title[:60]}...")
print(f"  XGBoost: {result_xgb_legit['prediction']} ({result_xgb_legit['confidence']:.2%} confidence)")
print(f"  TextCNN: {result_cnn_legit['prediction']} ({result_cnn_legit['confidence']:.2%} confidence)")
print("=" * 80)